# Human Activity Recognition using MHEALTH Dataset

## Objective
To classify human physical activities using accelerometer and gyroscope sensor data.

In [ ]:
# This script performs a comprehensive analysis of the MHEALTH dataset,
# including data loading, preprocessing, visualization, and training
# several machine learning models for activity recognition.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler, RobustScaler
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, confusion_matrix

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam

# --- Data Loading and Initial Exploration ---

# Load the dataset from the zipped CSV file.
data = pd.read_csv("../data/mhealth_raw_data.csv")

print("Initial data head:")
print(data.head())
print("\nData shape:")
print(data.shape)
print("\nMissing values per column:")
print(data.isna().sum())
print("\nDataFrame information:")
data.info()

# Select a subset of numeric columns for analysis.
numeric_fields = ["alx", "aly", "alz", "glx", "gly", "glz", "arx", "ary", "arz", "grx", "gry", "grz"]
numeric_data = data[numeric_fields]

print("\nSummary statistics for numeric columns:")
print(numeric_data.describe())
print(f"\nNumber of duplicate rows found: {data.duplicated().sum()}")


# --- Activity Mapping and Visualization ---

# Create a mapping dictionary for activity IDs to descriptive labels.
activity_mapping = {
    0: 'Nothing', 1: 'Standing still', 2: 'Sitting and relaxing', 3: 'Lying down',
    4: 'Walking', 5: 'Climbing stairs', 6: 'Waist bends forward', 7: 'Frontal elevation of arms',
    8: 'Knees bending (crouching)', 9: 'Cycling', 10: 'Jogging', 11: 'Running',
    12: 'Jump front & back'
}

# Select data for a single subject for plotting.
subject_one_data = data[data["subject"] == "subject1"]
sensor_types = ['a', 'g']

# Plot the sensor data for subject 1 for each activity.
for activity_id in range(1, 13):
    for sensor_type in sensor_types:
        print(f"Activity: {activity_mapping[activity_id]} - {'Accelerometer' if sensor_type == 'a' else 'Gyroscope'} data")

        current_activity_data = subject_one_data[subject_one_data['Activity'] == activity_id]

        plt.figure(figsize=(15, 4))

        # Plot left ankle sensor data
        plt.subplot(1, 2, 1)
        plt.plot(current_activity_data[sensor_type + 'lx'], label=sensor_type + 'lx')
        plt.plot(current_activity_data[sensor_type + 'ly'], label=sensor_type + 'ly')
        plt.plot(current_activity_data[sensor_type + 'lz'], label=sensor_type + 'lz')
        plt.title("Left Ankle Sensor")
        plt.legend()

        # Plot right wrist sensor data
        plt.subplot(1, 2, 2)
        plt.plot(current_activity_data[sensor_type + 'rx'], label=sensor_type + 'rx')
        plt.plot(current_activity_data[sensor_type + 'ry'], label=sensor_type + 'ry')
        plt.plot(current_activity_data[sensor_type + 'rz'], label=sensor_type + 'rz')
        plt.title("Right Wrist Sensor")
        plt.legend()

        plt.show()

# Visualize data distribution using a box plot.
plt.figure(figsize=(10, 6))
sns.boxplot(data=numeric_data)
plt.title("Distribution of Numeric Sensor Readings")
plt.xlabel("Sensor Axes")
plt.ylabel("Reading Value")
plt.show()

# --- Data Preprocessing: Balancing and Scaling ---

# Count the number of samples for each activity class.
class_counts = data["Activity"].value_counts()
print("\nNumber of samples per activity class:")
print(class_counts)

# Define a function to balance the dataset by downsampling each class
# to the size of the smallest class.
def balance_data_by_sampling(target_size):
    """
    Balances the dataset by downsampling all classes to the size of the
    smallest class, and then samples a total size based on the target.

    Args:
        target_size (int): The number of rows to sample from each class
                           before combining.

    Returns:
        pd.DataFrame: The balanced and sampled DataFrame.
    """
    activity_counts = data['Activity'].value_counts()
    min_count = activity_counts.min()
    balanced_df = data.groupby('Activity').apply(
        lambda x: x.sample(min_count, random_state=42)
    ).reset_index(drop=True)
    return balanced_df.sample(target_size * len(activity_counts), random_state=42)

# Sample the data to create a balanced subset for training.
columns_for_training = numeric_fields + ["Activity"]
sampled_data = balance_data_by_sampling(10342)
sampled_data = sampled_data[columns_for_training]

print("\nHead of the balanced and sampled data:")
print(sampled_data.head())

# Define features (X) and target (y).
X_features = sampled_data.iloc[:, :-1]
y_target = sampled_data.iloc[:, -1]
print(f"\nShape of the target variable (y): {y_target.shape}")

# Split the data into training and testing sets.
X_train, X_test, y_train, y_test = train_test_split(
    X_features, y_target, test_size=0.2, random_state=42, shuffle=True
)

# Convert to NumPy arrays and reshape the target for model compatibility.
X_train = np.asarray(X_train)
y_train = np.asarray(y_train).reshape(-1, 1)

X_test = np.asarray(X_test)
y_test = np.asarray(y_test).reshape(-1, 1)

print(f"\nTraining features shape: {X_train.shape}")
print(f"Training target shape: {y_train.shape}")
print("\nExample of training target data:")
print(y_train)

# Scale the data using RobustScaler, which is suitable for outliers.
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# --- Model Evaluation Function ---

def evaluate_model_performance(true_labels, predicted_labels):
    """
    Calculates and displays performance metrics and a confusion matrix
    for a given set of predictions.

    Args:
        true_labels (array): The ground truth labels.
        predicted_labels (array): The predicted labels from a model.
    """
    conf_matrix = confusion_matrix(true_labels, predicted_labels)
    # Convert confusion matrix to a binary heatmap for visualization
    binary_conf_matrix = np.where(conf_matrix > 1, 1, 0)

    # Plot the confusion matrix heatmap
    plt.figure(figsize=(13, 5))
    sns.heatmap(binary_conf_matrix, cmap="Purples", annot=True,
                xticklabels=list(activity_mapping.values()),
                yticklabels=list(activity_mapping.values()))
    plt.title("Confusion Matrix")
    plt.xlabel("Predicted Activities")
    plt.ylabel("Actual Activities")
    plt.show()

    # Calculate and print performance metrics
    accuracy = accuracy_score(true_labels, predicted_labels)
    precision = precision_score(true_labels, predicted_labels, average="macro")
    recall = recall_score(true_labels, predicted_labels, average="macro")
    f1 = f1_score(true_labels, predicted_labels, average="macro")

    print("--- Model Performance Metrics ---")
    print(f"Accuracy: {accuracy * 100:.2f} %")
    print(f"Precision: {precision * 100:.2f} %")
    print(f"Recall: {recall * 100:.2f} %")
    print(f"F1_Score: {f1 * 100:.2f} %")


# --- Model Training and Prediction ---

# 1. K-Nearest Neighbors (KNN) Classifier
print("\n\n--- Evaluating K-Nearest Neighbors Classifier ---")
knn_model = KNeighborsClassifier(n_neighbors=3)
knn_model.fit(X_train_scaled, np.ravel(y_train))
knn_predictions = knn_model.predict(X_test_scaled)
evaluate_model_performance(y_test, knn_predictions)

# 2. Support Vector Machine (SVC) Classifier
print("\n\n--- Evaluating Support Vector Machine Classifier ---")
svm_model = SVC(kernel="rbf", C=10)
svm_model.fit(X_train_scaled, np.ravel(y_train))
svm_predictions = svm_model.predict(X_test_scaled)
evaluate_model_performance(y_test, svm_predictions)

# 3. Logistic Regression Classifier
print("\n\n--- Evaluating Logistic Regression Classifier ---")
lr_model = LogisticRegression(C=6, max_iter=1000)
# Note: Using unscaled data for Logistic Regression to show a different approach,
# although scaling is generally recommended.
lr_model.fit(X_train, np.ravel(y_train))
lr_predictions = lr_model.predict(X_test)
evaluate_model_performance(y_test, lr_predictions)


Error: The file 'mhealth_raw_data.csv.zip' was not found. Please ensure it's in the correct directory.
Initial data head:


NameError: name 'data' is not defined

In [ ]:
import numpy as np



# [alx, aly, alz, glx, gly, glz]

new_data_sample = [-9.8, 0.5, 1.2, -0.1, 0.3, -0.2]
new_data = np.array([new_data_sample])



# You MUST use the 'scaler' object you already trained in your notebook.

new_data_scaled = scaler.transform(new_data)



# Use your best model (svm_model) to predict
new_prediction = svm_model.predict(new_data_scaled)
predicted_activity_number = new_prediction[0]

print(f"--- New Prediction ---")
print(f"The new sensor data was: {new_data_sample}")
print(f"The model predicts the activity is: {predicted_activity_number}")

